# 🎧 → 📄  Audiobook to Text (Google Colab)

Transcribe an audiobook (`.m4b`, `.mp3`, `.m4a`, `.wav`) to a text file — on Google's
free cloud computer with a **GPU**, so a 5-hour book takes **minutes**, and your own PC
does none of the heavy lifting.

**Just run each cell in order** (click the ▶ button on the left of a cell, or press
`Shift`+`Enter`). Read the short note above each one.

> **Notes.** (1) Your audio is uploaded to Google's servers to do this — fine for a book you
> own, but not fully private like the desktop version. (2) Keep this browser tab open until it
> finishes. (3) Big books are handled **one chapter at a time**, and the text is **saved as
> each chapter finishes** — so it stays light on memory and never loses progress.

## Step 0 — Turn on the free GPU  *(do this first)*

In the menu: **Runtime → Change runtime type → Hardware accelerator → `T4 GPU` → Save**.

Then run the cell below to confirm it's on.

In [ ]:
import subprocess
try:
    print(subprocess.check_output(['nvidia-smi', '-L']).decode().strip())
    print('\n✅ GPU is ON — transcription will be fast.')
except Exception:
    print('⚠️ No GPU yet. Do: Runtime → Change runtime type → T4 GPU → Save, then run this cell again.')

## Step 1 — Install the transcriber

Takes ~30 seconds. (Colab already has FFmpeg, so nothing else is needed.)

In [ ]:
!pip -q install faster-whisper
print('Installed. FFmpeg is already available on Colab.')

## Step 2 — Get your audiobook into Colab  *(pick ONE option)*

**Option A — Upload directly** (simplest; best for smaller files). Run the next cell,
click **Choose Files**, and pick your `.m4b`.

**Option B — From Google Drive** (best for big files, and it won't re-upload if you run
again). Skip Option A and use the Drive cell below instead.

In [ ]:
# OPTION A — upload straight from your computer.
# A big audiobook can be slow here; if it stalls, use Option B (Google Drive) below.
from google.colab import files
up = files.upload()
AUDIO_PATH = next(iter(up))          # the file you just picked
del up                               # free the in-memory copy (saves RAM)
print('Using:', AUDIO_PATH)

In [ ]:
# OPTION B — from Google Drive (run this INSTEAD of Option A).
# 1) First put your .m4b in your Google Drive (drive.google.com), e.g. a folder 'Audiobooks'.
# 2) Run this cell and allow access when asked.
# 3) Edit the path below to point at your file, then run again.
from google.colab import drive
drive.mount('/content/drive')
AUDIO_PATH = '/content/drive/MyDrive/Audiobooks/YOUR_FILE.m4b'   # <-- change this line
import os
print('Found file:' if os.path.isfile(AUDIO_PATH) else 'NOT found — fix the path:', AUDIO_PATH)

## Step 3 — Transcribe

This does the book **one chapter at a time** and writes each chapter to your text file as it
finishes, so it stays light on memory (no 'out of RAM' crashes) and never loses progress.

`MODEL` sets quality vs. speed: `tiny` (fastest) → `base` → `small` → `medium` → `large-v3`
(best). **`small`** is a great start; `medium` is fine here too. Run the cell and watch the
pieces tick by.

In [ ]:
MODEL = 'small'      # tiny | base | small | medium | large-v3  (medium is fine here too)
LANGUAGE = None      # None = auto-detect; or force e.g. 'en'
CHUNK_MINUTES = 20   # used only if the file has no chapters

import json, subprocess, textwrap, os, math
from faster_whisper import WhisperModel
import ctranslate2

try:
    AUDIO_PATH
except NameError:
    raise SystemExit('Set your file first: run the Option A upload cell OR the Option B Drive cell.')

def ffprobe_chapters(path):
    try:
        out = subprocess.check_output(['ffprobe','-v','quiet','-print_format','json','-show_chapters', path]).decode()
        res = []
        for i, c in enumerate(json.loads(out).get('chapters', [])):
            res.append((float(c.get('start_time', 0)), float(c.get('end_time', 0)),
                        (c.get('tags') or {}).get('title') or f'Chapter {i+1}'))
        return res
    except Exception:
        return []

def ffprobe_duration(path):
    try:
        out = subprocess.check_output(['ffprobe','-v','quiet','-print_format','json','-show_format', path]).decode()
        return float(json.loads(out)['format']['duration'])
    except Exception:
        return 0.0

# Build the list of pieces: one per chapter, or fixed-length chunks if there are none.
chapters = ffprobe_chapters(AUDIO_PATH)
if chapters:
    pieces = chapters
    print(f'Found {len(pieces)} chapter(s) — transcribing one at a time (low memory).')
else:
    dur = ffprobe_duration(AUDIO_PATH)
    n = max(1, math.ceil(dur / (CHUNK_MINUTES * 60)))
    pieces = [(i*CHUNK_MINUTES*60, min((i+1)*CHUNK_MINUTES*60, dur), None) for i in range(n)]
    print(f'No chapters — transcribing in {len(pieces)} piece(s) of ~{CHUNK_MINUTES} min.')

DEVICE = 'cuda' if ctranslate2.get_cuda_device_count() > 0 else 'cpu'
COMPUTE = 'float16' if DEVICE == 'cuda' else 'int8'
print('Loading model', MODEL, 'on', DEVICE, '— first time downloads it (~seconds)…')
model = WhisperModel(MODEL, device=DEVICE, compute_type=COMPUTE)

def wrap(t): return '\n'.join(textwrap.wrap(' '.join(t.split()), 100))

OUT_NAME = os.path.splitext(os.path.basename(AUDIO_PATH))[0] + '.txt'
open(OUT_NAME, 'w', encoding='utf-8').close()      # start a fresh file; we append as we go
PIECE = '/content/_piece.wav'

for idx, (cs, ce, title) in enumerate(pieces, 1):
    dur = max(0.0, ce - cs)
    # Cut just this piece to a small 16 kHz mono WAV (fast input-seek, low memory).
    subprocess.run(['ffmpeg','-y','-loglevel','error','-ss', f'{cs:.3f}','-i', AUDIO_PATH,
                    '-t', f'{dur:.3f}','-ac','1','-ar','16000','-c:a','pcm_s16le', PIECE], check=True)
    seg_iter, info = model.transcribe(PIECE, language=LANGUAGE, vad_filter=True)
    text = ' '.join(s.text.strip() for s in seg_iter).strip()
    with open(OUT_NAME, 'a', encoding='utf-8') as f:
        if title:
            f.write(f'\n## {title}\n\n')
        if text:
            f.write(wrap(text) + '\n\n')
    os.remove(PIECE)
    print(f'  piece {idx}/{len(pieces)} done', end='\r')

print('\nSaved:', OUT_NAME)

## Step 4 — Download your text file

Run this to save the `.txt` to your computer's **Downloads** folder. (If you used Google
Drive, the commented line also drops a copy into your Drive.)

In [ ]:
from google.colab import files
files.download(OUT_NAME)

# Google Drive users can also keep a copy in Drive:
# import shutil; shutil.copy(OUT_NAME, '/content/drive/MyDrive/'); print('Copied to your Drive.')

---
### Tips & troubleshooting
- **How long?** On the T4 GPU, expect a few minutes for a 5-hour book with `small`.
- **Ran out of RAM before?** This version transcribes one chapter at a time, so it won't.
- **Nothing downloaded?** The file is written as it goes; run Step 4 after Step 3 prints `Saved:`.
- **`No GPU` / very slow?** You skipped Step 0 — set the runtime to `T4 GPU` and run Steps 1–3 again.
- **Upload keeps failing?** Use Option B (Google Drive) instead of Option A.
- **Privacy:** for a fully-offline, nothing-leaves-your-PC run, use the desktop `aax2text` tool instead.